In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")


In [ ]:
df = pd.read_excel("final_health_data.xlsx")
df.head()

In [ ]:
print(df.dtypes)

# Get numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
print("\nNumerical columns:", list(numerical_cols))

In [ ]:
# Columns to scale (numerical ones from the specified list)
columns_to_scale = ['age', 'household_size', 'chronic_disease', 'hypertension', 'diabetes', 'asthma', 'missed_visits_12m', 'missed_screenings_12m', 'days_since_last_visit', 'followup_delay_days', 'screening_completed', 'vaccination_up_to_date', 'facility_access_km', 'region_health_index']

# Filter to ensure they are numerical
columns_to_scale = [col for col in columns_to_scale if col in numerical_cols]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

print("Scaled columns:", columns_to_scale)
print("Data scaled successfully.")

In [ ]:
df_scaled.head()

In [ ]:
# Create separate dataframes for each state
# Columns to exclude
exclude_cols = ['state_or_ut', 'entity_type', 'district', 'district_name']

# Get unique states
states = df_scaled['state_or_ut'].unique()
print(f"Total states/UTs: {len(states)}")

# Get all is_ columns
is_columns = [col for col in df_scaled.columns if col.startswith('is_')]
print(f"Found {len(is_columns)} is_ columns\n")

# Create a dictionary to store all state dataframes
state_dfs = {}

for state in states:
    # Filter data for this state
    state_df = df_scaled[df_scaled['state_or_ut'] == state].drop(columns=exclude_cols)
    
    # Create variable name (replace spaces and special characters with underscore)
    var_name = 'df_' + state.replace(' ', '_').replace('and', '').replace('&', '').lower()
    
    # Determine which is_ column belongs to this state
    state_is_col = None
    for is_col in is_columns:
        # Check if this is_ column corresponds to the current state
        col_name = is_col.replace('is_', '').replace('_', ' ')
        if col_name.lower() == state.lower():
            state_is_col = is_col
            break
    
    # Drop all is_ columns except the one for this state
    cols_to_drop = [col for col in is_columns if col != state_is_col]
    state_df = state_df.drop(columns=cols_to_drop)
    
    # Assign to globals (this creates the variables like df_maharashtra, df_assam, etc.)
    globals()[var_name] = state_df
    state_dfs[var_name] = state_df
    
    print(f"{var_name}: {state_df.shape[0]} records, kept column: {state_is_col if state_is_col else 'None'}")

print(f"\nCreated {len(state_dfs)} state dataframes")

In [ ]:
df_maharashtra.head()

In [ ]:
# Statistical tests to find relationship with illness
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler as SSC
from scipy.stats import pointbiserialr, chi2_contingency

print("="*70)
print("STATISTICAL ANALYSIS: Finding Features Related to Illness")
print("="*70)

# Get the target variable 'ill'
if 'ill' in df_scaled.columns:
    target = df_scaled['ill']
    print(f"\nTarget variable 'ill' distribution:\n{target.value_counts()}\n")
    
    # Get numerical features (excluding 'ill' and other non-feature columns)
    features = df_scaled.select_dtypes(include=[np.number]).columns
    features = [col for col in features if col != 'ill']
    
    # 1. POINT-BISERIAL CORRELATION (best for continuous vs binary)
    print("="*70)
    print("1. POINT-BISERIAL CORRELATION (Continuous vs Binary Outcome)")
    print("="*70)
    correlations = []
    for feature in features:
        corr, p_value = pointbiserialr(target, df_scaled[feature])
        correlations.append({
            'Feature': feature,
            'Correlation': corr,
            'P-Value': p_value,
            'Significant': 'Yes' if p_value < 0.05 else 'No'
        })
    
    corr_df = pd.DataFrame(correlations).sort_values('Correlation', key=abs, ascending=False)
    print(corr_df.head(15).to_string(index=False))
    
    # 2. INDEPENDENT T-TESTS (compare ill vs not ill groups)
    print("\n" + "="*70)
    print("2. INDEPENDENT T-TESTS (Ill vs Not Ill Groups)")
    print("="*70)
    t_tests = []
    for feature in features:
        ill_group = df_scaled[target == 1][feature].dropna()
        not_ill_group = df_scaled[target == 0][feature].dropna()
        
        t_stat, p_value = stats.ttest_ind(ill_group, not_ill_group)
        
        # Calculate Cohen's d (effect size)
        mean_diff = ill_group.mean() - not_ill_group.mean()
        pooled_std = np.sqrt((ill_group.std()**2 + not_ill_group.std()**2) / 2)
        cohens_d = mean_diff / pooled_std if pooled_std != 0 else 0
        
        t_tests.append({
            'Feature': feature,
            'T-Statistic': t_stat,
            'P-Value': p_value,
            'Mean_Ill': ill_group.mean(),
            'Mean_NotIll': not_ill_group.mean(),
            'Cohens_d': cohens_d,
            'Significant': 'Yes' if p_value < 0.05 else 'No'
        })
    
    t_test_df = pd.DataFrame(t_tests).sort_values('Cohens_d', key=abs, ascending=False)
    print(t_test_df[['Feature', 'T-Statistic', 'P-Value', 'Cohens_d', 'Significant']].head(15).to_string(index=False))
    
    # 3. LOGISTIC REGRESSION (feature importance)
    print("\n" + "="*70)
    print("3. LOGISTIC REGRESSION (Feature Coefficients - Importance)")
    print("="*70)
    X = df_scaled[features].fillna(df_scaled[features].mean())
    y = target
    
    log_reg = LogisticRegression(max_iter=1000, random_state=42)
    log_reg.fit(X, y)
    
    log_reg_df = pd.DataFrame({
        'Feature': features,
        'Coefficient': log_reg.coef_[0],
        'Abs_Coefficient': np.abs(log_reg.coef_[0])
    }).sort_values('Abs_Coefficient', ascending=False)
    
    print(log_reg_df.head(15).to_string(index=False))
    print(f"\nLogistic Regression R² Score: {log_reg.score(X, y):.4f}")
    
    print("\n" + "="*70)
    print("SUMMARY: Top 10 Features Most Related to Illness")
    print("="*70)
    top_features = corr_df.head(10)[['Feature', 'Correlation', 'P-Value']]
    print(top_features.to_string(index=False))
    
else:
    print("\n'ill' column not found in df_scaled")